# Iris Flower Classification and Machine Learning Model Comparison
**BCA – Artificial Intelligence | Google Colab Practical Project**

This notebook demonstrates dataset exploration, cleaning, preprocessing, model training, evaluation, clustering, and prediction using the Iris dataset.

**Algorithms:** Logistic Regression, Decision Tree, K-Nearest Neighbors (KNN), Artificial Neural Network (MLP), K-Means Clustering, and a separate Linear Regression demonstration.

> CNN, RNN, and LSTM are not included because the Iris dataset is small tabular data, not image or sequence data.

## 1. Import required libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, mean_squared_error, r2_score
)

print("All libraries imported successfully!")

## 2. Load the Iris dataset

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()

# Use readable feature names
df = df.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
    "target": "species_id"
})
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())

## 3. Explore the dataset

In [ ]:
print("Rows and columns:", df.shape)
print("\\nColumn information:")
df.info()

print("\\nStatistical summary:")
display(df.describe())

print("\\nSpecies counts:")
display(df["species"].value_counts())

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x="species", hue="species", legend=False)
plt.title("Number of Samples in Each Iris Species")
plt.xlabel("Species")
plt.ylabel("Count")
plt.show()

## 4. Data cleaning

In [ ]:
print("Missing values before cleaning:")
print(df.isnull().sum())

print("\\nDuplicate rows before cleaning:", df.duplicated().sum())

# Remove exact duplicate records
df = df.drop_duplicates().reset_index(drop=True)

print("\\nDataset shape after duplicate removal:", df.shape)
print("Missing values after duplicate removal:")
print(df.isnull().sum())

**Note:** The original Iris dataset is generally clean. We check missing values and duplicates rather than inventing errors. The preprocessing pipeline below also includes median imputation as a safeguard.

## 5. Prepare features and target

In [ ]:
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
X = df[feature_cols]
y = df["species_id"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Split completed successfully!")

## 6. Train classification models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(random_state=42))
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),
    "ANN (MLP)": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(hidden_layer_sizes=(10, 10), max_iter=2000, random_state=42))
    ])
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append({"Model": name, "Accuracy": accuracy})
    trained_models[name] = model
    print(f"{name}: trained successfully | Accuracy: {accuracy:.4f}")

results_df = pd.DataFrame(results).sort_values("Accuracy", ascending=False)
display(results_df)

## 7. Compare model accuracy

In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="Accuracy", y="Model", hue="Model", legend=False)
plt.xlim(0, 1.05)
plt.title("Classification Model Accuracy Comparison")
plt.xlabel("Accuracy")
plt.ylabel("Model")
plt.show()

## 8. Confusion matrix and classification report

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
best_predictions = best_model.predict(X_test)

print("Best model based on this test split:", best_name)
print("\\nClassification Report:")
print(classification_report(
    y_test, best_predictions,
    labels=[0, 1, 2],
    target_names=list(iris.target_names),
    zero_division=0
))

cm = confusion_matrix(y_test, best_predictions, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=iris.target_names)
disp.plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix – {best_name}")
plt.show()

## 9. K-Means clustering (unsupervised learning)

In [ ]:
# K-Means does not use the species labels during training.
scaler_kmeans = StandardScaler()
X_scaled = scaler_kmeans.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df["cluster"] = cluster_labels
print("K-Means clustering completed.")
print("Number of clusters:", len(np.unique(cluster_labels)))
display(df[["species", "cluster"]].head(10))

plt.figure(figsize=(7, 5))
sns.scatterplot(
    data=df, x="petal_length", y="petal_width",
    hue="cluster", palette="Set1"
)
plt.title("K-Means Clusters (Petal Features)")
plt.show()

**Interpretation:** K-Means groups records based on similarity in their feature values. Cluster IDs (0, 1, 2) are arbitrary and do not automatically correspond to species names.

## 10. Linear Regression demonstration (numerical prediction)

In [ ]:
# Linear Regression predicts a continuous number, not a flower class.
# Demonstration: predict petal length from the other three measurements.
X_reg = df[["sepal_length", "sepal_width", "petal_width"]]
y_reg = df["petal_length"]

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=42
)

linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LinearRegression())
])
linear_model.fit(Xr_train, yr_train)
reg_predictions = linear_model.predict(Xr_test)

print("Linear Regression trained successfully.")
print("Mean Squared Error:", round(mean_squared_error(yr_test, reg_predictions), 4))
print("R² Score:", round(r2_score(yr_test, reg_predictions), 4))

comparison = pd.DataFrame({
    "Actual petal length": yr_test.values,
    "Predicted petal length": reg_predictions
})
display(comparison.head())

## 11. Predict a new flower

In [ ]:
# Enter measurements in this order:
# sepal length, sepal width, petal length, petal width (all in cm)
new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=feature_cols
)

predicted_class = best_model.predict(new_flower)[0]
predicted_species = iris.target_names[predicted_class]

print("New flower measurements:")
display(new_flower)
print("Predicted species:", predicted_species)

## 12. Try your own flower measurements

In [ ]:
# Change these four values to test another flower.
my_flower = pd.DataFrame(
    [[6.0, 2.9, 4.5, 1.5]],
    columns=feature_cols
)
my_prediction = best_model.predict(my_flower)[0]
print("Your flower is predicted as:", iris.target_names[my_prediction])

## 13. Project conclusion

In [ ]:
print("PROJECT CONCLUSION")
print("------------------")
print("The Iris dataset was explored, checked for missing values and duplicates,")
print("preprocessed, and divided into training and testing sets.")
print("Four classification algorithms were trained and compared.")
print("K-Means clustering and a separate Linear Regression demonstration were also completed.")
print(f"The highest test accuracy in this run was from {best_name}: {results_df.iloc[0]['Accuracy']:.2%}.")
print("The trained classification model can predict the species of a new flower.")

## 14. Viva preparation

| Algorithm | Simple explanation |
|---|---|
| Linear Regression | Predicts a continuous numerical value by fitting a line. |
| Logistic Regression | Predicts a class using probabilities. |
| Decision Tree | Makes predictions using a sequence of feature-based questions. |
| KNN | Predicts using the classes of the nearest data points. |
| K-Means | Groups similar data points into clusters without labels. |
| ANN / MLP | Learns patterns through connected layers of artificial neurons. |

**Why not CNN, RNN, and LSTM here?**
- **CNN:** commonly used for images and spatial patterns.
- **RNN:** designed for sequential data.
- **LSTM:** a type of recurrent network that can learn longer-term sequence dependencies.

The Iris dataset is small tabular data, so these architectures are not the appropriate choice for this project.

## Final checklist
- [ ] All cells executed without errors
- [ ] Dataset cleaning and preprocessing shown
- [ ] Four classification models trained
- [ ] Accuracy comparison graph displayed
- [ ] Confusion matrix and classification report displayed
- [ ] K-Means clustering demonstrated
- [ ] Linear Regression numerical prediction demonstrated
- [ ] New flower prediction tested
- [ ] Conclusion and viva concepts reviewed

**Submission tip:** In Google Colab, use `Runtime → Run all` after checking the notebook, then `File → Download → Download .ipynb` to save your completed notebook.